# 05 · Unit, prediction point, predictors and leakage gate — Fetal death, MIMIC-IV (M3 · R3)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 · G1 → 02: S4 → 03: S5 · S6 → R3 → 04: R3 · S2 · S3 · G1 · S4 → **05: S5 · S6 · S7 · G2** → 06: G3 · class · S11 |
| What it does | Builds or checks the antenatal predictor table (one row per admission and margin), shows the unit and the grouping by patient (S5), the information window of each margin (S6), the 27 candidate parameters with their coverage and missing values (S7), and checks that nothing of the current admission enters the model (G2) |
| Stages and gates | S5 · S6 · S7 · G2 |
| Decisions | 27 candidate parameters and the reduced set of 17, fixed before any model; multiple imputation fitted inside each training split; see `docs/outcome_card_antenatal.md`, sections 3 to 5 |
| Inputs | `antenatal_obstetric_universe` and `antenatal_fetal_death_label` (notebook 04) |
| Tables of the mimiciv schema | `antenatal_predictors_v1` (`sql/03_antenatal/04_predictor_table.sql`) |
| Outputs | `data/outputs_05_antenatal_predictors.txt` (aggregate counts only; 1 to 4 shown as "<5") |
| Time | About a minute when the table exists; building it takes several minutes |
| Previous / next | `04_antenatal_count_audit` / `06_sample_size_gate` |
| Spanish version | `05_antenatal_predictors_es.ipynb` (same code) |

> **Rebuild the tables.** The tables of the `mimiciv` schema that this notebook needs are read as they are, and only their checks run. If a table does not exist, it is built from its SQL script. To rebuild it (for example after a change in a script of `sql/`), set **`REBUILD = True`** in the first code cell and run the notebook again.

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_05_antenatal_predictors.txt"
engine = get_connect_database()
from libraries.antenatal_design import BLOCKS, REDUCED_SET, REDUCED_PARAMETERS, EXCLUDED_COLUMNS, design_matrix
from sklearn.model_selection import StratifiedGroupKFold
SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5

## S5 · Unit of analysis — one record per delivery admission, grouped by patient

The predictor table has one row per admission of the population and margin. Every validation split groups the admissions of the same patient on one side; the cell checks it on the splits that gate G3 uses (notebook 06).

| Result | What is done | In this model |
|---|---|---|
| One row per admission and no patient on both sides of any split | On to S6 | 19,440 admissions of 13,735 patients; 0 patients shared |
| Duplicated rows or a patient on both sides | Review the table or the splits before going on | — |

In [ ]:
# Stage S5: predictor table (one row per admission and margin) and grouping by patient
resp = ensure_table(engine, "antenatal_predictors_v1", SQL / "03_antenatal" / "04_predictor_table.sql", REBUILD, log=say)
df = pd.read_sql(f"SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = {BUFFER_DAYS}", engine)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
per_patient = df.groupby("subject_id").size()
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
shared = [len(set(groups[tr]) & set(groups[te])) for tr, te in cv.split(np.zeros(len(y)), y, groups)]
show("S5 · Unit of analysis and grouping (7-day margin)", pd.DataFrame([{
    "n_admissions": len(df), "n_distinct_admissions": df["hadm_id"].nunique(), "n_patients": df["subject_id"].nunique(),
    "n_patients_with_more_than_one": int((per_patient > 1).sum()), "n_strict_events": int(y.sum()),
    "n_patients_shared_between_train_and_test": int(sum(shared))}]))
s5_ok = df["hadm_id"].is_unique and len(df) == 19440 and sum(shared) == 0
decision("S5 · Unit of analysis", [
    ("record", f"{len(df):,} rows, one per admission" if df["hadm_id"].is_unique else "DUPLICATED rows",
     "one record per delivery hospitalization"),
    ("grouping of the 5 splits of gate G3", f"{sum(shared)} patients on both sides", "every split grouped by patient")])
assert s5_ok, "S5: the unit or the grouping is not as expected"

## S6 · Prediction point and information window of each margin

Prediction point: the admission time. Information window: everything recorded at least B days before (B = 7 main; 1 and 30 as sensitivity). Measurements and laboratory values of the current pregnancy are taken between 280 and 7 days before the admission. Of the current admission only attributes that do not depend on how the patient arrived are used (age, race and ethnicity, insurance, language, marital status).

| Result | What is done | In this model |
|---|---|---|
| Every margin keeps the population and the events of G1 | The prediction point is fixed: on to S7 | 19,440 rows and 129 events in each margin |
| A margin loses rows or events | Review `sql/03_antenatal/04_predictor_table.sql` | — |

In [ ]:
# Stage S6: rows, patients and events of each information margin (Q-P1 of the predictor script)
show_results(resp, ["Q-P1 integrity"])
p1 = resp["Q-P1 integrity"]
s6_ok = bool(((p1.n_rows == 19440) & (p1.n_admissions == 19440) & (p1.n_strict_events == 129)).all())
decision("S6 · Prediction point", [
    ("prediction point", "admission time", "information recorded at least 7 days before (1 and 30 days as sensitivity)"),
    ("current pregnancy", "measurements and laboratory values", "only between 280 and 7 days before the admission"),
    ("margins 1, 7 and 30 days", "population and events kept" if s6_ok else "ROWS OR EVENTS LOST", "on to S7" if s6_ok else "review the table")])

## S7 · Predictors — the 27 candidate parameters, their coverage and their missing values

The predictors were fixed with the frequencies of the count audit (whole population, never split by outcome) and clinical reasons, before any model. The design matrix is the one of `libraries/antenatal_design.py`, the same that notebook 06 uses. Missing values are not imputed here: multiple imputation is fitted inside the training part of each split.

| Result | What is done | In this model |
|---|---|---|
| 27 parameters, as in the outcome card | On to G2; with fewer than 20 events per parameter the main model is a penalized logistic regression | 27 parameters; 129 events (4.8 per parameter) |
| Missing values in the measurements of the current pregnancy | Multiple imputation inside each training split; the indicator of prenatal care in this hospital marks who has records | Blood pressure 38.7%, hemoglobin 41.2%, body mass index 57.3% |
| Gate G3 could need fewer parameters | Reduced set of 17 fixed now, never chosen by association with the outcome | Used only if G3 failed with 27 |

In [ ]:
# Stage S7: coverage of the predictors (whole population) and the design matrix of the 27 candidate parameters
show_results(resp, ["Q-P2 categorical and binary predictors (7 days)", "Q-P3 continuous predictors (7 days)",
                    "Q-P4 body mass index source (7 days)"])
X = design_matrix(df)
P = X.shape[1]
blocks = pd.DataFrame([(b, ", ".join(cols), len(cols)) for b, cols in BLOCKS.items()],
                      columns=["block", "parameters of the design matrix", "n_parameters"])
show("S7 · Candidate parameters by block", blocks)
miss = X.isna().sum()
show("S7 · Missing values of the design matrix (whole population)",
     pd.DataFrame({"n_missing": miss[miss > 0], "pct_missing": (100 * miss[miss > 0] / len(X)).round(1)}).rename_axis("parameter"),
     index=True)
show("S7 · Reduced set fixed before any model (used only if gate G3 failed with the 27 parameters)",
     pd.DataFrame({"predictor": REDUCED_SET}))
decision("S7 · Predictors", [
    ("candidate parameters", f"{P}", "as in the outcome card" if P == 27 else "DIFFERS from the outcome card"),
    ("events per candidate parameter", f"{y.sum() / P:.1f}", "below 20: main model, penalized logistic regression"),
    ("missing values", "only in the measurements of the current pregnancy", "multiple imputation inside each training split"),
    ("reduced set", f"{REDUCED_PARAMETERS} parameters", "fixed before any model, for gate G3")])
assert P == 27, P

## G2 · Leakage gate — nothing of the current admission and nothing after the window

A predictor must be known at least 7 days before the admission, or be an attribute of the patient that does not depend on how she arrived. Admission type and location are excluded because the event itself can determine them (a woman with a known fetal death may come for an induction). Diagnoses, procedures, laboratory values and emergency data of the current admission never enter. Every split is grouped by patient (S5).

| Result | What is done | In this model |
|---|---|---|
| No excluded column in the design matrix and no patient on both sides | G2 passes: on to gate G3 (notebook 06) | Passes |
| An excluded column is in the matrix or a patient is on both sides | Remove it or fix the splits before G3 | — |

In [ ]:
# Gate G2: leakage checks
in_matrix = [c for c in EXCLUDED_COLUMNS if c in df.columns or c in X.columns]
g2 = pd.DataFrame([
    ("columns of the current admission in the predictor table or the design matrix (admission type or location, discharge, emergency, DRG)",
     ", ".join(in_matrix) or "none"),
    ("diagnoses, procedures or laboratory values of the current admission", "none: the table only reads earlier admissions and outpatient records"),
    ("measurements of the current pregnancy", "between 280 and 7 days before the admission"),
    ("patients on both sides of the 5 splits", int(sum(shared))),
], columns=["check", "result"])
show("G2 · Leakage checks", g2)
g2_ok = not in_matrix and sum(shared) == 0
decision("G2 · Leakage gate", [("information of the current admission or after the window", "none" if g2_ok else "FOUND",
                                "passes: on to gate G3 (notebook 06)" if g2_ok else "DOES NOT PASS: fix before G3")])
assert g2_ok, "gate G2 does not pass"

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 05
notebook_summary([
    ("admissions / patients (7-day margin)", lambda: f"{len(df):,} / {df['subject_id'].nunique():,}"),
    ("strict events", lambda: int(y.sum())),
    ("candidate parameters", lambda: P),
    ("events per parameter", lambda: round(y.sum() / P, 1)),
    ("missing: blood pressure / hemoglobin / body mass index", lambda: f"{100 * miss['last_sbp'] / len(X):.1f}% / {100 * miss['last_hemoglobin'] / len(X):.1f}% / {100 * miss['last_bmi'] / len(X):.1f}%"),
    ("gate G2", lambda: "passes" if g2_ok else "DOES NOT PASS"),
    ("next step", "sample size gate (G3), notebook 06"),
])
write_outputs(OUT_FILE, "Notebook 05 (S5, S6, S7, G2), fetal death MIMIC-IV, antenatal prediction point; aggregate output only")